In [ ]:
# 1. Install necessary libraries
!pip install flickrapi folium pandas textblob

import flickrapi
import pandas as pd
import folium
import time
from textblob import TextBlob
from google.colab import files
from IPython.display import display

# 2. Enter your Flickr API credentials
API_KEY = ''
API_SECRET = ''

# Initialize Flickr API
flickr = flickrapi.FlickrAPI(API_KEY, API_SECRET, format='parsed-json')

# 3. Set search parameters for Castlefield, Manchester
castlefield_bbox = '-2.2625,53.4695,-2.2490,53.4785'

# Your updated list of landscape and infrastructure keywords
castlefield_keywords = [
    'castlefield', 'canal', 'canal basin', 'basin', 'waterside',
    'warehouse', 'wharf', 'marina', 'railway arches', 'arches',
    'viaduct', 'towpath', 'lock', 'bridge', 'walk', 'view',
    'urban landscape', 'historic building', 'waterfront', 'canalside'
]
tags_string = ','.join(castlefield_keywords)

print(f"Searching Flickr for the following keywords: {tags_string}")
print(f"Using precise bounding box: {castlefield_bbox}")
print("Starting auto-pagination fetch for Castlefield with sentiment analysis...")

try:
    # 4. Use a loop for auto-pagination
    all_photos_dict = {}
    current_page = 1
    max_pages = 20

    while current_page <= max_pages:
        print(f"Fetching data for page {current_page}...")

        photos = flickr.photos.search(
            tags=tags_string,
            tag_mode='any',
            bbox=castlefield_bbox,
            has_geo=1,
            extras='geo,url_s,description',
            per_page=250,
            page=current_page
        )

        photo_list = photos['photos']['photo']

        if not photo_list:
            print("No more photos, fetching completed.")
            break

        for photo in photo_list:
            # Safely extract title and description
            title = photo.get('title', '')

            desc_dict = photo.get('description', {})
            description = desc_dict.get('_content', '') if isinstance(desc_dict, dict) else ''

            # Combine title and description for sentiment analysis
            combined_text = f"{title} {description}".strip()

            # Calculate polarity using TextBlob (-1.0 to 1.0)
            if combined_text:
                sentiment = TextBlob(combined_text).sentiment
                polarity = sentiment.polarity
            else:
                polarity = 0.0 # Default to neutral

            # Save the polarity back into the photo dictionary
            photo['polarity'] = polarity

            all_photos_dict[photo['id']] = photo

        total_pages = int(photos['photos']['pages'])
        if current_page >= total_pages:
            print(f"Reached the last page ({total_pages}), fetching completed.")
            break

        current_page += 1
        time.sleep(1)

    final_photo_list = list(all_photos_dict.values())
    print(f"\nSuccess! Fetched {len(final_photo_list)} unique photos related to Castlefield.")

    # 5. Convert data to Pandas DataFrame and clean
    if len(final_photo_list) > 0:
        df = pd.DataFrame(final_photo_list)
        df['latitude'] = df['latitude'].astype(float)
        df['longitude'] = df['longitude'].astype(float)

        df = df[(df['latitude'] != 0.0) & (df['longitude'] != 0.0)]

        # 6. Create pure dark interactive map with no labels
        print("Generating sentiment color-coded map...")

        castlefield_map = folium.Map(
            location=[53.4740, -2.2558],
            zoom_start=16,
            tiles='https://{s}.basemaps.cartocdn.com/dark_nolabels/{z}/{x}/{y}{r}.png',
            attr='&copy; OpenStreetMap contributors &copy; CARTO'
        )

        # Add each photo as a colored dot based on polarity
        for idx, row in df.iterrows():
            polarity_value = row['polarity']

            # Assign color based on polarity value
            if polarity_value > 0:
                point_color = 'green'  # Positive
            elif polarity_value < 0:
                point_color = 'red'    # Negative
            else:
                point_color = 'blue'   # Neutral (exactly 0)

            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=2,
                color=point_color,
                weight=1,
                fill=True,
                fill_color=point_color,
                fill_opacity=0.8
            ).add_to(castlefield_map)

        # 7. Export HTML and CSV files and auto-download
        print("Preparing to download files to your computer...")

        html_filename = 'castlefield_sentiment_map.html'
        castlefield_map.save(html_filename)

        csv_filename = 'castlefield_sentiment_data.csv'

        # --- 关键修改部分：现在的 CSV 导出只提取这三列纯数据 ---
        export_df = df[['latitude', 'longitude', 'polarity']]
        export_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        # -----------------------------------------------------

        files.download(html_filename)
        files.download(csv_filename)

        print("All done! Please check your browser's download pop-ups.")
        print("Map preview below (Green = Positive, Red = Negative, Blue = Neutral):")

        display(castlefield_map)

    else:
        print("No photos found matching these keywords in Castlefield.")

except flickrapi.exceptions.FlickrError as e:
    print(f"API call error: {e}")
    print("Please ensure your API_KEY and API_SECRET are correct.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Searching Flickr for the following keywords: castlefield,canal,canal basin,basin,waterside,warehouse,wharf,marina,railway arches,arches,viaduct,towpath,lock,bridge,walk,view,urban landscape,historic building,waterfront,canalside
Using precise bounding box: -2.2625,53.4695,-2.2490,53.4785
Starting auto-pagination fetch for Castlefield with sentiment analysis...
Fetching data for page 1...
Fetching data for page 2...
Fetching data for page 3...
Fetching data for page 4...
Fetching data for page 5...
Fetching data for page 6...
Fetching data for page 7...
Fetching data for page 8...
Fetching data for page 9...
Fetching data for page 10...
Fetching data for page 11...
Fetching data for page 12...
Fetching data for page 13...
Fetching data for page 14...
Fetching data for page 15...
Fetching data for page 16...
Fetching data for page 17...
Fetching data for page 18...
Fetching data for page 19...
Reached the last page (18), fetching completed.

Success! Fetched 3472 unique photos related to

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All done! Please check your browser's download pop-ups.
Map preview below (Green = Positive, Red = Negative, Blue = Neutral):
